# 欢迎来到第 2 天实验！—— Ollama 与开源模型

## 练习目标（理念）

理解 **Chat Completions API** 与 **OpenAI 兼容端点**：同一套 `OpenAI` 客户端，只要换 `base_url` / `api_key` / `model`，就能打 Google Gemini 或本地 Ollama。

- 先用 HTTP / SDK 调 Gemini
- 再连本地 Ollama（`llama3.2`、`deepseek-r1:1.5b`）
- 最后作业：用 Ollama + Selenium 抓 Sephora 首页做摘要推荐

## 怎么跑

1. 需要时可在 `.env` 配 `GEMINI_API_KEY` / `GOOGLE_API_KEY`
2. 本地部分：安装并启动 Ollama，`ollama pull llama3.2`
3. 从上到下运行；不想用 Gemini 可跳过相关格


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">开始之前——</h2>
            <span style="color:#f71;">我想先指向课程的实用资源页，其中包含全部幻灯片链接。<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            请收藏此页，我会持续补充更多有用链接。
            </span>
        </td>
    </tr>
</table>


## 首先——聊聊 Chat Completions API

1. 调用 LLM 最简单的方式
2. 之所以叫 Chat Completions，是因为它在说：「这是一段对话，请预测接下来该说什么」
3. Chat Completions API 由 OpenAI 发明，但因太流行几乎人人都在用！

### 我们先再次调用——不用担心非 OpenAI 用户，马上轮到你们！

下面先读 `GEMINI_API_KEY`，再用「裸 HTTP」打 Google 的 OpenAI 兼容端点，建立对 Endpoint 的直觉。


In [ ]:
# ========== 加载 GEMINI_API_KEY 并做有无检查 ==========

# 导入标准库 os：读环境变量
import os
# 从 dotenv 导入 load_dotenv：把 .env 读进进程环境
from dotenv import load_dotenv

# override=True：文件值覆盖已有同名环境变量
load_dotenv(override=True)
# 取出 Gemini Key（本格用的名字是 GEMINI_API_KEY）
gemini_api_key = os.getenv('GEMINI_API_KEY')

# 没有 key 就提示；有就确认（英文提示保持原样）
if not gemini_api_key:
    print("No Gemini API key found. Please check your .env file.")
else:
    print("Gemini API key found!")


## 你知道什么是 Endpoint（端点）吗？

若不清楚，请复习 guides 文件夹里的 Technical Foundations 指南。

端点 =「你把 JSON 请求 POST 到哪个 URL」。下一格先拼好 headers 与 payload，再下一格真正 `requests.post`。


In [ ]:
# ========== 拼 HTTP 请求：Authorization + Chat Completions JSON ==========

# 导入 requests：后面用 POST 打 Google 的兼容端点
import requests

# headers：Bearer 令牌 + JSON Content-Type（密钥来自上一格）
headers = {"Authorization": f"Bearer {gemini_api_key}", "Content-Type": "application/json"}

# payload：model id 与 messages；发给模型的 user 内容保持英文
payload = {
    "model": "gemini-2.5-flash",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

# 笔记本里直接写 payload：展示将要发送的 JSON 结构
payload


In [ ]:
# ========== 裸 HTTP：POST 到 Google 的 OpenAI 兼容 Chat Completions 端点 ==========

# URL 是厂商提供的兼容端点；不要改路径拼写
response = requests.post(
    "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions",
    headers=headers,
    json=payload
)

# 把响应体解析成 Python dict 看看完整结构
response.json()


In [ ]:
# ========== 从 JSON 里取出助手回复正文 ==========

# 与 OpenAI 相同路径：choices[0].message.content
response.json()["choices"][0]["message"]["content"]


# openai 包是什么？

它是一个 Python 客户端库。

本质上只是对这个 HTTP 端点调用的一层封装。

让你用干净的 Python 代码，而不用折腾难看的 JSON 对象。

仅此而已。开源且轻量。有人以为它内含 OpenAI 模型代码——其实没有！

下一格用同一个 Gemini Key，但改走 `OpenAI(base_url=...)` SDK。


In [ ]:
# ========== 用 OpenAI SDK 打 Gemini 兼容端点（同一 fun fact）==========

# 从 openai 导入 OpenAI 客户端类
from openai import OpenAI

# base_url 指向 Google 的 OpenAI 兼容根路径；api_key 用 Gemini Key
gemini = OpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)
# 与裸 HTTP 等价：model + messages
response = gemini.chat.completions.create(
    model="gemini-2.5-flash",
    messages=[
        {"role": "user", "content": "Tell me a fun fact"}
    ]
)

# SDK 路径下取正文：response.choices[0].message.content
response.choices[0].message.content


## 然后发生了很棒的事：

OpenAI 的 Chat Completions API 太受欢迎，其他模型厂商也做了相同形态的端点。

它们被称为「OpenAI Compatible Endpoints」（OpenAI 兼容端点）。

例如 Google 做了一个： https://generativelanguage.googleapis.com/v1beta/openai/

OpenAI 也很慷慨：你可以直接用他们为 GPT 做的同一个客户端库，只需指定不同的端点 URL 与 Key，就能调用其他厂商。

例如你可以这样写：

```python
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="AIz....")
gemini.chat.completions.create(...)
```

说清楚：代码里虽有 OpenAI，但我们只用这个轻量 Python 客户端去打端点——这里并不涉及 OpenAI 模型。

若仍困惑，请复习 Guides 文件夹中的 Guide 9！

现在来试试！

## 这部分可选——若想试用 Google Gemini，请访问：

https://aistudio.google.com/

并在此创建 API Key：

https://aistudio.google.com/api-keys

然后把 Key 写入 `.env`，改完后务必保存：

`GOOGLE_API_KEY=AIz...`

注意：上一格用的是 `GEMINI_API_KEY`；下面可选路径改读 `GOOGLE_API_KEY`（以 `AIz` 开头）。


In [ ]:
# ========== 可选：检查 GOOGLE_API_KEY（AIz 前缀）==========

# Gemini OpenAI 兼容 API 的根地址常量
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# 再加载一次 .env，确保刚写入的 GOOGLE_API_KEY 生效
load_dotenv(override=True)

# 取出 Google AI Studio 的 Key
google_api_key = os.getenv("GOOGLE_API_KEY")

# 校验：缺失 / 前缀不对 / 看起来 OK——英文提示保持原样
if not google_api_key:
    print("No API key was found - please be sure to add your key to the .env file, and save the file! Or you can skip the next 2 cells if you don't want to use Gemini")
elif not google_api_key.startswith("AIz"):
    print("An API key was found, but it doesn't start AIz")
else:
    print("API key found and looks good so far!")


In [ ]:
# ========== 可选：用 GOOGLE_API_KEY + gemini-2.5-flash-lite 再问一句 ==========

# 指向 Google 兼容端点的客户端
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)

# 换一个更轻的模型 id；user 内容保持英文
response = gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=[{"role": "user", "content": "Tell me a fun fact"}])

# 取出回复正文
response.choices[0].message.content


## Ollama 也提供 OpenAI 兼容端点

……而且跑在你自己的机器上！

若下一格没有打印 `Ollama is running`，请打开终端运行 `ollama serve`。


In [ ]:
# ========== 探测本机 Ollama 是否在跑（默认 11434 端口）==========

# GET 根路径；正常时应返回类似 b'Ollama is running' 的内容
requests.get("http://localhost:11434").content


### 从 Meta 下载 llama3.2

若电脑配置较低，可改成 `llama3.2:1b`。

不要用 llama3.3 或 llama4！对你的电脑来说通常太大……

下一格用 Jupyter 的 `!` shell magic 执行 `ollama pull`。


In [ ]:
# Jupyter shell magic：拉取本地模型 llama3.2（需已安装 Ollama）
!ollama pull llama3.2


In [ ]:
# ========== 创建指向本地 Ollama 的 OpenAI 兼容客户端 ==========

# Ollama 的 OpenAI 兼容 API 根地址（注意是 /v1）
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# api_key 对本地通常不校验，占位字符串 'ollama' 即可
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# ========== 用本地 llama3.2 要一个 fun fact ==========

# Get a fun fact：model 名须与 ollama pull 一致；prompt 英文保持原样
response = ollama.chat.completions.create(model="llama3.2", messages=[{"role": "user", "content": "Tell me a fun fact"}])

# 打印/展示助手正文
response.choices[0].message.content


In [ ]:
# ========== 再试 deepseek-r1:1.5b（需本机已 pull 该模型）==========

# 同一客户端、换 model 字符串即可；若未 pull 会报错
response = ollama.chat.completions.create(model="deepseek-r1:1.5b", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content


In [ ]:
# 现在拉取 deepseek-r1:1.5b——这是蒸馏进阿里云 Qwen 的 DeepSeek 小模型
# Jupyter shell magic：ollama pull 指定标签
!ollama pull deepseek-r1:1.5b


# 家庭作业练习

把第 1 天的网页摘要项目升级为：通过 Ollama 在本地跑开源模型，而不是 OpenAI。

若不想用付费 API，后续项目也都能用这套技巧。

**优点：**
1. 无 API 费用——开源
2. 数据不离开本机

**缺点：**
1. 能力明显弱于前沿模型

## 回顾 Ollama 安装

访问 [ollama.com](https://ollama.com) 安装即可！

安装完成后，ollama 服务通常已在本地运行。  
若你访问：  
[http://localhost:11434/](http://localhost:11434/)

你应该看到消息 `Ollama is running`。  

如果没有，请打开新的 Terminal（Mac）或 Powershell（Windows），输入 `ollama serve`。  
再在另一个 Terminal / Powershell 输入 `ollama pull llama3.2`。  
然后再次打开 [http://localhost:11434/](http://localhost:11434/)。

若 Ollama 在你的机器上偏慢，可改用 `llama3.2:1b`：在终端执行 `ollama pull llama3.2:1b`，并把代码里的 `MODEL = "llama3.2"` 改成 `MODEL = "llama3.2:1b"`。

下一格完整示例：Selenium 抓 Sephora 首页 → 本地 llama3.2 做「傲慢美妆顾问」摘要推荐。


In [ ]:
# ========== 作业示例：Selenium 抓 Sephora + 本地 Ollama 摘要推荐 ==========

# 导入 OpenAI 客户端（打本地兼容端点）
from openai import OpenAI
# Selenium：无头/有头浏览器抓 JS 站点
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
# BeautifulSoup：从渲染后的 HTML 抽文本
from bs4 import BeautifulSoup
# Markdown：在笔记本里渲染回答
from IPython.display import Markdown
# time.sleep：等待页面 JS
import time

# 本地模型名：需已 ollama pull；机器慢可改成 llama3.2:1b
MODEL = "llama3.2"

# 指向本机 Ollama /v1；api_key 占位
ollama = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
# Chrome 启动选项（此处未强制 headless，保持原逻辑）
options = Options()
driver = webdriver.Chrome(options=options)

# 打开 Sephora 首页（URL 决定抓取目标）
driver.get("https://www.sephora.com")
# 等待页面渲染
time.sleep(5)
# 取渲染后的完整 HTML
html = driver.page_source

# 解析 HTML
soup = BeautifulSoup(html, "html.parser")

# 去掉 script/style，减少噪音
for tag in soup(["script", "style"]):
    tag.decompose()

# 抽出可见文本
text = soup.get_text(separator="\n", strip=True)

# 关闭浏览器
driver.quit()

# system prompt：傲慢美妆顾问角色（英文指令不翻译）
system_prompt = """
You are an arrogant beauty advisor.
Summarize the Sephora homepage and recommend interesting products.
Return the answer in markdown.
"""

# user：截断正文前 12000 字符，避免上下文过长
user_prompt = text[:12000]
# 调用本地模型：system + user
response = ollama.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
)
# 以 Markdown 展示回答（笔记本最后一式表达式会显示）
Markdown(response.choices[0].message.content)
